<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **4. Visible Light Communication**


<div class="alert alert-light">

  **Name:** Stefan Sykes
  
  **Matriculation Number:** 11-250-693
  
</div>



### 4.1 Introduction

Visible light communication (VLC) transmits data by varying the intensity of a light source, usually a light-emitting diode (LED). A receiver detects those changes and reconstructs the data. The modulation can be fast enough that the light appears steady to the human eye.

Li-Fi, short for light fidelity, refers to networking through light. VLC and Li-Fi are related terms, but they are not interchangeable: VLC specifies visible wavelengths, while light-based networks can also use infrared. For example, IEEE 802.11bb specifies communication in the 800–1,000 nm infrared band [\[05\]](#ref05). This notebook explores a simple LED-to-LED VLC link, rather than a complete Wi-Fi replacement.

An optical link depends on the light reaching the receiver. Alignment, distance, obstacles, ambient light, and reflections affect reception. Opaque walls can restrict coverage, but physical confinement alone does not guarantee security. A dark room does not prevent communication if the transmitter still emits light; infrared communication does not require visible illumination.

An LED can detect light as well as emit it. Firmware can switch it between transmitting and receiving, allowing a single LED to serve both roles at different times. A receiving circuit can detect light-induced changes in the LED's electrical behavior [\[01\]](#ref01). This reduces component count, but practical range and data rate depend on the LED, circuit, and firmware. Such links are useful for exploring inexpensive communication between small devices.

<center>
  <img src="../../.admin/notebooks/picture/vlc_toys_banner.png" 
       alt="VLC Toys" 
       width="80%"
       style="filter: grayscale(80%);">
  <p><em>Interactive toys with low cost LED-to-LED communication.</em></p>
</center>

The experiments use two preprogrammed Arduino boards to control LEDs, exchange messages, and measure communication range and round-trip time. One board is enough for the initial LED tests; communication requires a second board, connected either to the same computer or to a partner's computer. The exercises compare LED colors, acknowledged and broadcast messages, and different payload sizes.


### 4.2 References


[<a id="ref01">01</a>] P. Dietz, W. Yerazunis, and D. Leigh. [Very Low-Cost Sensing and Communication Using Bidirectional LEDs](https://www.merl.com/publications/docs/TR2003-35.pdf). MERL Technical Report TR2003-35, July 2003.

[<a id="ref02">02</a>] S. Schmid, G. Corbellini, S. Mangold, and T. R. Gross. [Continuous Synchronization for LED-to-LED Visible Light Communication Networks](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/VLC,%20Visible%20Light%20Communication/2014-continuous-synchronization-led-to-led-visible-light-networks.pdf). 3rd International Workshop on Optical Wireless Communications (IWOW), September 2014, Madeira, Portugal.

[<a id="ref03">03</a>] S. Schmid, G. Corbellini, S. Mangold, and T. R. Gross. [LED-to-LED Visible Light Communication Networks](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/VLC,%20Visible%20Light%20Communication/2013-led-to-led-visible-light-communication-networks.pdf). ACM MobiHoc, 29 July–1 August 2013, Bangalore, India.

[<a id="ref04">04</a>] ETH Zurich, Wireless group. [VLC background reading](https://gitlab.ethz.ch/wireless/BackgroundReading/-/tree/main/VLC,%20Visible%20Light%20Communication). Course reading collection.

[<a id="ref05">05</a>] IEEE. [IEEE Std 802.11bb-2023: Light Communications](https://standards.ieee.org/ieee/802.11bb/10823/). 2023.

[<a id="ref06">06</a>] Arduino. [UNO R3](https://docs.arduino.cc/hardware/uno-rev3/). Board documentation.

[<a id="ref07">07</a>] pySerial. [Overview and installation](https://pyserial.readthedocs.io/en/latest/pyserial.html). Python serial-port library documentation.


### 4.3 Experiments


#### 4.3.1 Preparing the microcontroller boards

<div class="alert alert-block alert-info">Form a team of two participants. Each team receives two Arduino Uno boards, two USB cables, two red LEDs, and two blue LEDs. Return all equipment after the session.</div>

**Arduino Uno R3:** The Arduino Uno R3 is a microcontroller board based on the ATmega328P. It provides 14 digital input/output pins, six analog inputs, and a 16 MHz clock [\[06\]](#ref06). USB supplies power and a serial connection to the computer. The supplied boards already contain the VLC firmware, so the experiments do not require the Arduino IDE or firmware installation.

**ATmega328P microcontroller:** The ATmega328P executes the board's firmware. It has 32 KB of flash memory for program storage, 2 KB of SRAM for working data, and 1 KB of EEPROM for persistent data [\[06\]](#ref06). Its input/output pins and timers allow the firmware to control LED emission and sample received signals. The computer sends commands over USB; the microcontroller handles the optical communication.

**General-purpose input/output pins:** A general-purpose input/output (GPIO) pin can drive a digital voltage as an output or read a voltage as an input. On the Uno R3, a high output is approximately 5 V and a low output is approximately 0 V. Ordinary LED circuits need current limiting; the experiments here use the supplied boards with their prepared VLC firmware.

Firmware can change a pin's mode during operation. This allows an LED circuit to alternate between light emission and sensing. Although A0 and A1 are labeled as analog inputs, they can also operate as digital GPIO pins. Those are the two connections used in this experiment.

<center>
<img src="../../.admin/notebooks/picture/arduino_uno_r3.png" 
     alt="Arduino" 
     width="40%" 
     style="filter: grayscale(40%);">
<img src="../../.admin/notebooks/picture/LED.png" 
     alt="LED" 
     width="15%" 
     style="filter: grayscale(40%);">
     <p><em>Arduino Uno R3 and one LED. The prepared VLC setup uses connectors A0 and A1.</em></p>
</center>

<div class="alert alert-block alert-info">
<p>Place each board on a nonconductive surface, such as cardboard or acrylic, and use USB power for these experiments. Disconnect USB before inserting, removing, or reversing an LED. Keep metal tools, loose wires, and liquids away from powered boards. If a board becomes hot or behaves unexpectedly, disconnect it and ask the support team to check the wiring.</p>
<p>With USB disconnected, insert one red LED into each supplied board: cathode to A0 and anode to A1. On an untrimmed LED, the anode is usually the longer lead; the flat edge of the housing usually marks the cathode. Ask the support team if polarity is unclear. Reconnect USB after checking the placement. A lit LED indicates emission, but does not by itself confirm that communication works.</p>
</div>

For two boards on one computer, the notebook uses local serial ports A and B. With one board on each of two computers, each notebook uses its own port A. The optical device addresses must still differ, regardless of the local port labels.


#### 4.3.2 Control console

The console manages serial connections, LED tests, and optical messages. The cell below checks pySerial and detects the connected USB boards before displaying the controls. The Python check and complete port report appear in the general log below the Arduino panels. Connect the boards before running it; rerun it after adding or removing a board. Two selected boards appear side by side with independent controls and logs; one selected board uses the full width. The [VLC serial protocol reference](../../03_Assignments/VlcSerial.md) explains the commands and events shown in the logs.

<div class="alert alert-block alert-info">Run the code cell below and wait for the control console to appear. The console may take a few seconds to appear.</div>


In [5]:
from pathlib import Path
import runpy

helper = next(p for p in (
    Path("../../.admin/notebooks/lifi/control_console.py"),
    Path(".admin/notebooks/lifi/control_console.py"),
) if p.is_file())
vlc_console = runpy.run_path(str(helper.resolve()))["show_console"](globals())


#### 4.3.3 Getting acquainted: three checks

1. **Connections and LEDs:** The first log entries show the detected USB ports, followed by each board’s connection status and initial settings. A short startup pause is normal because opening a serial connection can reset the board. If a board is missing, checking its USB connection and rerunning the console repeats detection.
   
   Blink LED offers a simple first test: the selected board should blink three times, with the corresponding commands appearing in its log. This confirms USB control and LED operation; optical communication is explored next. The LED ON/OFF switch changes the light-emission setting without closing the connection. Blink LED restores the selected setting afterward. Incoming messages remain visible in the log; compare reception with the switch in both positions to check the firmware’s behavior.

2. **Addresses and settings:** With the LEDs facing each other at a short distance, the default addresses `AB` and `CD` provide a starting point. Each board’s Send to field should match the other board’s Address. With one board per computer, both consoles call their local board Arduino A, so the optical addresses still need to be different: `AB` on one computer and `CD` on the other.
   
   Changing an Address, Retries, or FEC value and pressing Enter provides a quick way to observe configuration commands and their echoes in the board’s log. Leaving the field also applies the change. Send to selects the destination for the next message. The initial values of three retries and a 30-byte FEC threshold are suitable for getting acquainted with the controls.

3. **Messages and reception:** Send unicast transmits the message to the selected destination. With a working optical link, the message should appear in the receiving board’s log, and the sender should report an acknowledgment (ACK). Send broadcast offers a useful comparison: the receiver can show the same message, but no acknowledgment is expected. A TX entry alone confirms that a command was sent over USB; the receiving log provides evidence of optical delivery.
   
   Repeat broadcast sends a message approximately every two seconds, making it easier to observe reception while gently changing the LED alignment or briefly blocking the light path. Received messages should become less frequent or stop when the link is obstructed, and resume when reception improves. Stop broadcast ends this stream before the later measurements. An edited message takes effect after restarting repeated broadcasting.




---
### 4.4 Questions

<div class="alert alert-block alert-info">This notebook can be worked on as a team. Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work. Some questions are marked with "(NO HARDWARE NEEDED)" - they can be worked on later at home, as they do not require a dedicated measurement. </div>



**With two red LEDs, what is the greatest tested distance at which repeated messages are still received? Start at about 1 cm and record the approximate distance, alignment, and lighting conditions.**

>Use two red LEDs for the initial measurements. Check the transmitter port, receiver address, payload size, and packet count before starting. Keep LED alignment and lighting as consistent as possible when comparing distances or payload sizes. Packet events and timing output appear in the control console. Point the red LEDs directly at each other, starting about 1 cm apart. Either connect both boards to one computer or use one board per computer. The transmitting board sends data and receives acknowledgments; the receiving board must be configured with the destination address used by the test.
>
> <center>
> <img src="../../.admin/notebooks/picture/vlc_measurement.png" 
>      alt="VLC Measurement" 
>      width="40%" 
>      style="filter: grayscale(40%);">
>      <p><em>Measurement setup. Record the approximate distance between the LEDs, for example 10 cm.</em></p>
> </center>

Answer:
- Transmitter port: COM3
- Receiver address: AB
- Payload size: 1 * 10 Bytes
- Packet count: 10
- Distance: Until about 20 cm apart the devices receive data.

**What round-trip times are observed with 1-byte and 180-byte payloads at the same distance? Explain any difference using the packet exchange rather than the speed of light alone.**

> Run the performance test with Bytes/packet set to 1, then 180, keeping the distance and other settings unchanged. Compare the round-trip times shown in the console’s general log.

Answer:
- Test 1 Setup: Interval = 1 second; bytes/packet=1; packets=10 => Results: 10 packets submitted; 10 matched ACKs in 9.4 seconds; mean RTT 202.2 ms; 1,07 packets/second; 8.5 payload bit/s.
- Test 2 Setup: Interval = 1 second; bytes/packet=180; packets=10 => Results: 10 packets submitted; 5 matched ACKs in 19.2 seconds; mean RTT 1764 ms; 0.26 packets/second; 374.6 payload bit/s.
- Overall Result: The 1-byte payload had a mean RTT of 202.2 ms, while the 180-byte payload had a mean RTT of 1,764 ms (1.764 seconds). The larger payload therefore had a substantially higher RTT, and only 5 of the 10 packets received matched acknowledgements, compared with all 10 for the 1-byte payload.

Conclusion: This difference cannot be explained by the speed of light alone, since the distance remained unchanged. Larger packets take longer to transmit and may require more processing. The missing acknowledgements and lower packet throughput also suggest delays or losses during the packet exchange. These results indicate that packet size and protocol behaviour affect performance, although more repeated tests would be needed to determine whether the difference is consistent.


**(NO HARDWARE NEEDED) In the previous task, how many RTT samples were recorded in each run? Is that enough to describe typical performance reliably? Explain how more samples and a measure of variability would improve the comparison. Consider the concept of a confidence interval.**

Answer: In the first run, 10 RTT samples were recorded, as all 10 packets received matched acknowledgements. In the second run, only 5 RTT samples were recorded because only 5 matched acknowledgements were received.

These sample sizes are relatively small and may not reliably represent typical performance. Repeating each test with more packets would provide more RTT samples and improve the reliability of the comparison. Calculating the standard deviation would show how much the RTT values vary. A 95% confidence interval around the mean RTT would indicate the uncertainty of the estimated average and help assess whether the observed difference between the two payload sizes is statistically meaningful.


**Replace only the transmitter's red LED with a blue LED. Does the observed range change? Compare reception of broadcasts with unicast.**

Answer:
- Test 1 Setup: Interval = 1 second; bytes/packet=1; packets=10 => Results: 10 packets submitted; 10 matched ACKs in 9.3 seconds; mean RTT 202.0 ms; 1,07 packets/second; 8.6 payload bit/s.
- Test 2 Setup: Interval = 1 second; bytes/packet=180; packets=10 => Results: 10 packets submitted; 5 matched ACKs in 19.2 seconds; mean RTT 1762.4 ms; 0.26 packets/second; 374.4 payload bit/s.
- Overall Result: Replacing only the transmitter’s red LED with a blue LED did not noticeably change the observed communication performance. The RTT values remained almost identical to the previous test.

Conclusion: Broadcasts can be received by all devices within range, whereas unicast communication is intended for a specific receiver. Broadcast reception depends on whether devices can detect the transmitted signal, while unicast also requires successful communication with the intended recipient and may involve acknowledgements. Therefore, the observed range may differ depending on the receiver and communication method, but changing the LED colour alone did not produce a significant change in the recorded results.


**Do two blue LEDs communicate? How does their observed range compare with two red LEDs? For a mixed-color pair, test both transmission directions and record any difference.**

Answer: 
- Two blue LEDs can communicate if the receiver can detect the transmitted blue light.
- The observed range with two blue LEDs is much smaller (~5 cm), than with red LEDs. This is because the receiver LED's sensitivity depends on the wavelength of the incoming light. A red LED used as a receiver is generally more sensitive to red light than to blue light, and vice versa. If the LED is less sensitive to the transmitted wavelength, the signal becomes harder to detect over longer distances.
- Transmitter with blue LED, receiver with red LED: In this case the range is even smaller than before (~2 cm).
- Transmitter with red LED, receiver with blue LED: When the red LED transmits and the blue LED receives, no data is received. This suggests that the blue LED is not sufficiently sensitive to the red light wavelength to detect the signal reliably. As a result, the received signal is too weak to decode, and communication fails. 

Conclusion: This demonstrates that communication range depends not only on distance but also on the compatibility between the transmitting and receiving LEDs. 


**Can a change to the receiver threshold or FEC settings improve range?.**

Answer: Yes, adjusting the receiver threshold or FEC (Forward Error Correction) settings improves the observed communication range. Lowering the receiver threshold can help detect weaker signals over longer distances, but it may also increase noise and false detections. Enabling or increasing FEC adds redundancy to the transmitted data, allowing the receiver to correct some errors and potentially decode data at weaker signal levels. However, FEC may reduce the effective data rate. These changes can improve communication reliability, but they cannot compensate for a receiver that is fundamentally insensitive to the transmitted wavelength.


**(NO HARDWARE NEEDED) How do infrared and visible light differ in wavelength and human visibility?**

Answer: Infrared light has a longer wavelength than visible light (above 650 nm) and is invisible to the human eye. Visible light has wavelengths of approximately 400–700 nm and can be seen by humans, ranging from violet to red.


**(NO HARDWARE NEEDED) How does water attenuate infrared and visible light? Explain why wavelength and water conditions matter.**

Answer: Water absorbs and scatters both infrared and visible light, but infrared light is generally absorbed much more strongly by water, so it travels a shorter distance. Visible light, especially blue light, can travel farther through clear water. The attenuation depends on wavelength because water absorbs different wavelengths at different rates, and on water conditions such as turbidity, suspended particles, and dissolved substances, which increase scattering and absorption.


**(NO HARDWARE NEEDED) What are the benefits and limitations of using an LED as a receiver instead of a dedicated photodiode?**

Answer: An LED can act as a light receiver because it generates a small electrical current when light of a suitable wavelength hits it. Its main benefits are low cost, simplicity, and the ability to use the same component for both transmitting and receiving light. However, an LED is generally less sensitive than a dedicated photodiode, produces a weaker signal, and may have a narrower detection range of wavelengths. Therefore, a photodiode is usually better for reliable, long-distance, or low-light communication.

**OPTIONAL: Are all tasks in this notebook clear and understandable? Are there any shortcomings, missing parts, or sections that should be improved?**

(add feedback or suggestions for improvement here)
